In [1]:
!pip install transformers datasets


In [2]:
# HW5 - Question 2: Diffusion-style Large Language Models for Text-to-SQL

# This notebook implements the full pipeline requested in Q2 of Assignment 5:
# - Data loading and normalization
# - Prompt construction in chat format
# - Forward masking noise model
# - Training loop for a masked-iterative Text-to-SQL model
# - Block diffusion sampling for generation
# - Post-processing, evaluation (normalized EM), and basic error analysis

print("Notebook for HW5 Q2 initialized.")


Notebook for HW5 Q2 initialized.


In [3]:
import os
import math
import random
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from transformers import AutoTokenizer, AutoModelForMaskedLM, get_linear_schedule_with_warmup
except ImportError:
    raise ImportError("Please install transformers: pip install transformers")

try:
    from datasets import load_dataset
except ImportError:
    raise ImportError("Please install datasets: pip install datasets")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [4]:
# -----------------------------
# Configuration
# -----------------------------

class Config:
    # Model
    # For faster training in HW, you can switch to a smaller model like "distilbert-base-uncased"
    model_name = "distilbert-base-uncased"  # masked LM backbone
    max_length = 256

    # Training
    batch_size = 16
    num_epochs = 1          # keep small for runtime; increase for better quality
    lr = 3e-5
    warmup_ratio = 0.1

    # Optional: limit number of examples used from the big Gretel dataset
    # Use small subsets so the HW notebook runs quickly.
    max_train_examples = 5000    # set to None to use full train set
    max_dev_examples = 500
    max_test_examples = 500

    # Forward masking / diffusion parameters
    num_steps = 8              # number of diffusion steps during training
    mask_schedule_power = 2.0  # controls how fast masking increases (forward masking)

    # Loss reweighting over diffusion steps (section 2.1.3)
    # weight(t) ∝ (t / num_steps) ** loss_weight_power
    loss_weight_power = 1.0

    # Sampling / block diffusion
    gen_length = 160           # number of answer tokens after the prompt
    block_length = 40          # tokens per block
    sample_steps = 8           # iterative refinement steps per block
    commit_fraction = 0.25     # fraction of remaining masks to commit at each step

cfg = Config()


In [5]:
# -----------------------------
# Data utilities: loading + normalization
# -----------------------------

SQL_KEYWORDS = [
    "SELECT", "FROM", "WHERE", "GROUP", "BY", "ORDER", "LIMIT", "JOIN",
    "INNER", "LEFT", "RIGHT", "FULL", "ON", "HAVING", "UNION", "INTERSECT",
    "EXCEPT", "WITH", "UPDATE", "DELETE", "INSERT", "INTO", "VALUES"
]


def normalize_sql(sql: str) -> str:
    """Rough SQL normalization for EM evaluation.

    - Strip whitespace
    - Uppercase keywords
    - Collapse multiple spaces
    - Remove trailing semicolon
    """
    if sql is None:
        return ""
    s = sql.strip()
    if s.endswith(";"):
        s = s[:-1]
    # Normalize spaces
    s = " ".join(s.split())
    tokens = s.split(" ")
    norm_tokens = []
    for t in tokens:
        up = t.upper()
        if up in SQL_KEYWORDS:
            norm_tokens.append(up)
        else:
            norm_tokens.append(t)
    return " ".join(norm_tokens)


def normalized_em(pred: str, gold: str) -> float:
    return float(normalize_sql(pred) == normalize_sql(gold))


@dataclass
class Example:
    question: str
    schema: str
    sql: str


def _infer_field_name(keys, candidates, field_type: str, required: bool = True) -> Optional[str]:
    """Heuristically pick a column name from a list of candidates.

    This lets us adapt to different dataset schemas without crashing.
    """
    key_set = set(keys)
    for cand in candidates:
        if cand in key_set:
            return cand
    if required:
        raise KeyError(f"Could not find any of {candidates} for {field_type} in dataset columns {keys}")
    return None


def hf_split_to_examples(split) -> List[Example]:
    """Convert a HuggingFace dataset split (e.g., ds["train"]) to Example objects.

    This version is tailored for gretelai/synthetic_text_to_sql, whose columns are:
    - sql_prompt   : natural language question
    - sql_context  : database schema / context
    - sql          : target SQL query
    """
    if len(split) == 0:
        return []

    sample = split[0]
    keys = list(sample.keys())
    print("Available columns in dataset:", keys)

    # For this dataset, map explicitly
    q_col = "sql_prompt"
    schema_col = "sql_context"
    sql_col = "sql"

    out: List[Example] = []
    for row in split:
        question = row[q_col]
        sql = row[sql_col]
        schema = row[schema_col]
        out.append(Example(question=question, schema=schema, sql=sql))
    return out


print("Data utilities ready. Using HuggingFace dataset gretelai/synthetic_text_to_sql.")


Data utilities ready. Using HuggingFace dataset gretelai/synthetic_text_to_sql.


In [6]:
# -----------------------------
# Prompt construction in chat format
# -----------------------------

SYSTEM_PROMPT = (
    "You are a Text-to-SQL assistant. "
    "Given a natural language question about a database and its schema, "
    "you must output ONLY a valid SQL query that correctly answers the question."
)


def build_chat_prompt(ex: Example) -> str:
    """Build a chat-style prompt: [SYSTEM] + [USER] asking for SQL.

    We keep the answer zone (SQL) as the model's continuation.
    """
    user_msg = (
        "You are given a database schema and a question. "
        "Return ONLY the SQL query that answers the question.\n\n"
        f"Schema:\n{ex.schema}\n\n"
        f"Question: {ex.question}\n"
        "SQL:"
    )
    # Simple chat-style formatting; you can adapt to your framework
    prompt = f"[SYSTEM] {SYSTEM_PROMPT}\n[USER] {user_msg}\n[ASSISTANT] "
    return prompt


# Quick sanity check with a dummy example
_dummy = Example(question="How many students are there?", schema="table students(id, name)", sql="SELECT COUNT(*) FROM students")
print(build_chat_prompt(_dummy))


[SYSTEM] You are a Text-to-SQL assistant. Given a natural language question about a database and its schema, you must output ONLY a valid SQL query that correctly answers the question.
[USER] You are given a database schema and a question. Return ONLY the SQL query that answers the question.

Schema:
table students(id, name)

Question: How many students are there?
SQL:
[ASSISTANT] 


In [7]:
# -----------------------------
# Tokenization and dataset
# -----------------------------

TOKENIZER = AutoTokenizer.from_pretrained(cfg.model_name)
if TOKENIZER.mask_token is None:
    # Ensure we have a [MASK] token
    TOKENIZER.add_special_tokens({"mask_token": "[MASK]"})


class Text2SQLDataset(Dataset):
    def __init__(self, examples: List[Example]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.examples[idx]
        prompt = build_chat_prompt(ex)
        full_text = prompt + " " + ex.sql
        enc = TOKENIZER(
            full_text,
            max_length=cfg.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}  # input_ids, attention_mask
        # For training with masking, labels will be created in the collate function
        item["full_text"] = full_text
        item["sql"] = ex.sql
        return item


def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    full_texts = [b["full_text"] for b in batch]
    sqls = [b["sql"] for b in batch]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "full_texts": full_texts,
        "sqls": sqls,
    }


print("Tokenizer and dataset classes are ready.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer and dataset classes are ready.


In [8]:
# -----------------------------
# Forward masking noise model
# -----------------------------

MASK_TOKEN_ID = TOKENIZER.mask_token_id
PAD_TOKEN_ID = TOKENIZER.pad_token_id


def forward_mask_schedule(t: int, num_steps: int, power: float = 2.0) -> float:
    """Fraction of tokens that should be masked at step t (1..num_steps).

    This is a "forward" schedule: masking increases over time.
    We use a simple power schedule: (t / num_steps)^power.
    """
    return (t / num_steps) ** power


def apply_forward_masking(
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    t: int,
    num_steps: int,
    power: float = 2.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Apply forward masking to input_ids at step t.

    Returns (noisy_ids, mask_positions) where mask_positions is a boolean tensor
    indicating where we applied [MASK].
    """
    bsz, seq_len = input_ids.shape
    frac = forward_mask_schedule(t, num_steps, power)

    noisy = input_ids.clone()
    mask_positions = torch.zeros_like(noisy, dtype=torch.bool)

    # For each sequence, randomly choose a subset of non-pad tokens to mask
    for i in range(bsz):
        valid_positions = (attention_mask[i] == 1) & (input_ids[i] != PAD_TOKEN_ID)
        idxs = torch.nonzero(valid_positions, as_tuple=False).view(-1)
        n_to_mask = max(1, int(len(idxs) * frac))
        if n_to_mask >= len(idxs):
            chosen = idxs
        else:
            perm = idxs[torch.randperm(len(idxs))]
            chosen = perm[:n_to_mask]
        noisy[i, chosen] = MASK_TOKEN_ID
        mask_positions[i, chosen] = True

    return noisy, mask_positions


# Sanity check on a dummy batch
ids = torch.randint(low=5, high=1000, size=(2, 10))
attn = torch.ones_like(ids)
noisy_ids, mask_pos = apply_forward_masking(ids, attn, t=3, num_steps=cfg.num_steps, power=cfg.mask_schedule_power)
print("Forward masking example:")
print("orig:", ids)
print("noisy:", noisy_ids)
print("mask_pos:", mask_pos)


Forward masking example:
orig: tensor([[ 26, 310, 125, 254, 217, 836, 582, 659, 464, 354],
        [477, 204, 119, 716, 583, 929, 716, 793, 447, 819]])
noisy: tensor([[ 26, 103, 125, 254, 217, 836, 582, 659, 464, 354],
        [477, 204, 119, 716, 103, 929, 716, 793, 447, 819]])
mask_pos: tensor([[False,  True, False, False, False, False, False, False, False, False],
        [False, False, False, False,  True, False, False, False, False, False]])


In [9]:
# -----------------------------
# Model, training objective and loop (masked iterative training)
# -----------------------------

MODEL = AutoModelForMaskedLM.from_pretrained(cfg.model_name)
if hasattr(MODEL, "resize_token_embeddings"):
    MODEL.resize_token_embeddings(len(TOKENIZER))
MODEL.to(DEVICE)

optimizer = torch.optim.AdamW(MODEL.parameters(), lr=cfg.lr)


def _loss_weight_for_t(t: int) -> float:
    """Reweighting factor for timestep t (2-1-3 in the assignment).

    Later (noisier) steps can be up-weighted by choosing loss_weight_power > 1.
    """
    frac = t / cfg.num_steps
    return float(frac ** cfg.loss_weight_power)


def train_step(batch: Dict[str, Any], step_t: int) -> float:
    MODEL.train()
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    noisy_ids, mask_positions = apply_forward_masking(
        input_ids, attention_mask, t=step_t, num_steps=cfg.num_steps, power=cfg.mask_schedule_power
    )

    labels = input_ids.clone()
    # Only compute loss on masked positions; set others to -100
    labels[~mask_positions] = -100

    outputs = MODEL(
        input_ids=noisy_ids,
        attention_mask=attention_mask,
        labels=labels,
    )
    base_loss = outputs.loss

    # Reweight depending on diffusion step
    weight = _loss_weight_for_t(step_t)
    loss = base_loss * weight

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    return float(loss.item())


def train_model(train_ds: Text2SQLDataset, dev_ds: Optional[Text2SQLDataset] = None):
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )

    total_steps = cfg.num_epochs * len(train_loader) * cfg.num_steps
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    global_step = 0
    for epoch in range(cfg.num_epochs):
        for batch in train_loader:
            # Sample a random diffusion step for this batch (like training in DDPM)
            step_t = random.randint(1, cfg.num_steps)
            loss = train_step(batch, step_t)
            scheduler.step()
            global_step += 1
            if global_step % 50 == 0:
                print(f"Epoch {epoch+1}, step {global_step}, t={step_t}, loss={loss:.4f}")

    print("Training finished.")


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [10]:
# -----------------------------
# Block diffusion sampling with confidence-based committing
# -----------------------------

@torch.no_grad()
def masked_iterative_generate(
    prompt_text: str,
    gen_length: int,
    block_length: int,
    num_steps: int,
    commit_fraction: float,
) -> str:
    """Block-by-block diffusion-style masked generation.

    - We first tokenize the prompt and then append gen_length [MASK] tokens.
    - We split the answer region into blocks of size block_length.
    - For each block, we run num_steps refinement steps:
        * Predict tokens for all masked positions in the current block
        * Compute confidence = probability of chosen token
        * Commit top-k fraction (by confidence) of remaining masks
    """
    MODEL.eval()

    # Ensure that prompt length + generation length never exceeds the
    # underlying model's maximum positional embedding size (e.g. 512 for DistilBERT).
    # This avoids runtime errors like:
    #   "The size of tensor a (557) must match the size of tensor b (512) at non-singleton dimension 1"
    max_model_len = getattr(TOKENIZER, "model_max_length", 512)
    # Cap the prompt tokens so that prompt_len + gen_length <= max_model_len
    max_prompt_len = min(cfg.max_length, max_model_len - gen_length)

    enc = TOKENIZER(
        prompt_text,
        max_length=max_prompt_len,
        truncation=True,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    bsz, src_len = input_ids.shape
    assert bsz == 1

    # Extend sequence with [MASK] tokens for the answer
    answer_ids = torch.full((1, gen_length), MASK_TOKEN_ID, dtype=torch.long, device=DEVICE)
    input_ids = torch.cat([input_ids, answer_ids], dim=1)
    answer_mask = torch.ones_like(answer_ids)
    attention_mask = torch.cat([attention_mask, answer_mask], dim=1)

    total_len = input_ids.shape[1]
    start_answer = src_len
    end_answer = src_len + gen_length

    for block_start in range(start_answer, end_answer, block_length):
        block_end = min(block_start + block_length, end_answer)
        for step in range(num_steps):
            outputs = MODEL(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # (1, seq_len, vocab)
            probs = logits.softmax(dim=-1)

            # focus on current block
            block_slice = slice(block_start, block_end)
            block_ids = input_ids[0, block_slice]
            block_probs = probs[0, block_slice]  # (block_len, vocab)

            # positions that are still [MASK]
            mask_positions = (block_ids == MASK_TOKEN_ID)
            if not mask_positions.any():
                break

            masked_idx = torch.nonzero(mask_positions, as_tuple=False).view(-1)
            masked_logits = block_probs[masked_idx]  # (n_masked, vocab)

            # greedy proposal
            top_tokens = masked_logits.argmax(dim=-1)  # (n_masked,)
            top_probs = masked_logits.softmax(dim=-1).max(dim=-1).values  # confidence

            # how many to commit this step?
            n_remaining = len(masked_idx)
            n_commit = max(1, int(math.ceil(commit_fraction * n_remaining)))
            # select indices with highest confidence
            sorted_idx = torch.argsort(top_probs, descending=True)
            commit_idx = sorted_idx[:n_commit]

            # apply commits
            for pos in commit_idx:
                seq_pos = block_start + masked_idx[pos].item()
                token_id = top_tokens[pos].item()
                input_ids[0, seq_pos] = token_id

    # Decode only the answer part
    answer_tokens = input_ids[0, start_answer:end_answer]
    # stop at first pad or unused mask
    valid = []
    for tid in answer_tokens.tolist():
        if tid in (PAD_TOKEN_ID, MASK_TOKEN_ID):
            break
        valid.append(tid)
    decoded = TOKENIZER.decode(valid, skip_special_tokens=True)
    return decoded.strip()


# Quick dry-run with the dummy prompt
sample_sql = masked_iterative_generate(build_chat_prompt(_dummy), cfg.gen_length, cfg.block_length, cfg.sample_steps, cfg.commit_fraction)
print("Sample generated SQL (untrained model, will be junk):", sample_sql)


Sample generated SQL (untrained model, will be junk): 


In [11]:
# -----------------------------
# Post-processing to extract SQL from raw text
# -----------------------------

SQL_START_KEYWORDS = [
    "SELECT", "WITH", "UPDATE", "DELETE", "INSERT"
]


def extract_sql_from_text(text: str) -> str:
    """Keep only the SQL span.

    - Drop any text before the first SQL keyword
    - Optionally cut at the first ';'
    """
    if not text:
        return ""
    up = text.upper()
    start_idx = None
    for kw in SQL_START_KEYWORDS:
        pos = up.find(kw)
        if pos != -1:
            if start_idx is None or pos < start_idx:
                start_idx = pos
    if start_idx is None:
        return text.strip()
    sql_part = text[start_idx:]
    # Cut at first semicolon
    semi = sql_part.find(";")
    if semi != -1:
        sql_part = sql_part[:semi]
    return sql_part.strip()


# Quick examples
print(extract_sql_from_text("Here is the answer: SELECT * FROM t; Thanks"))
print(extract_sql_from_text("SELECT x"))


SELECT * FROM t
SELECT x


In [12]:
# -----------------------------
# Evaluation loop: normalized EM and error analysis
# -----------------------------

@torch.no_grad()
def evaluate_model(test_ds: Text2SQLDataset, n_print: int = 10) -> Dict[str, Any]:
    loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    em_scores: List[float] = []
    examples_to_show: List[Dict[str, str]] = []

    for i, batch in enumerate(loader):
        full_text = batch["full_texts"][0]
        gold_sql = batch["sqls"][0]

        # Build prompt again from the first part (up to the "SQL:" marker)
        # Because full_text = prompt + " " + sql, we can reconstruct prompt by cutting sql
        if gold_sql in full_text:
            prompt_text = full_text[: full_text.rfind(gold_sql)].strip()
        else:
            prompt_text = full_text

        raw_out = masked_iterative_generate(
            prompt_text,
            gen_length=cfg.gen_length,
            block_length=cfg.block_length,
            num_steps=cfg.sample_steps,
            commit_fraction=cfg.commit_fraction,
        )
        pred_sql = extract_sql_from_text(raw_out)
        em = normalized_em(pred_sql, gold_sql)
        em_scores.append(em)

        if len(examples_to_show) < n_print:
            examples_to_show.append(
                {
                    "prompt": prompt_text,
                    "gold_sql": gold_sql,
                    "raw_out": raw_out,
                    "pred_sql": pred_sql,
                    "em": em,
                }
            )

    mean_em = float(sum(em_scores) / max(1, len(em_scores)))
    print(f"Normalized EM on test set: {mean_em:.4f} over {len(em_scores)} examples")
    return {"em": mean_em, "examples": examples_to_show}


print("Evaluation utilities ready. You can now wire everything together below.")


Evaluation utilities ready. You can now wire everything together below.


In [13]:
# -----------------------------
# Main: load data, train, evaluate
# -----------------------------

if __name__ == "__main__":
    # Load HuggingFace dataset
    raw_ds = load_dataset("gretelai/synthetic_text_to_sql")

    # Map splits to our Example objects
    train_examples = hf_split_to_examples(raw_ds["train"])
    dev_split_name = "validation" if "validation" in raw_ds else "test"
    dev_examples = hf_split_to_examples(raw_ds[dev_split_name])
    test_examples = hf_split_to_examples(raw_ds["test"])

    # Optionally subsample for faster HW runtime
    if cfg.max_train_examples is not None:
        train_examples = train_examples[: cfg.max_train_examples]
    if cfg.max_dev_examples is not None:
        dev_examples = dev_examples[: cfg.max_dev_examples]
    if cfg.max_test_examples is not None:
        test_examples = test_examples[: cfg.max_test_examples]

    train_ds = Text2SQLDataset(train_examples)
    dev_ds = Text2SQLDataset(dev_examples)
    test_ds = Text2SQLDataset(test_examples)

    print(f"Train / Dev / Test sizes (after subsampling): {len(train_ds)} / {len(dev_ds)} / {len(test_ds)}")

    # Train model
    train_model(train_ds, dev_ds)

    # Evaluate
    results = evaluate_model(test_ds, n_print=10)
    for i, ex in enumerate(results["examples"]):
        print("\n=== Example", i, "===")
        print("PROMPT:\n", ex["prompt"])
        print("GOLD SQL:\n", ex["gold_sql"])
        print("RAW OUT:\n", ex["raw_out"])
        print("PRED SQL:\n", ex["pred_sql"])
        print("EM:", ex["em"])


README.md: 0.00B [00:00, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Available columns in dataset: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']
Available columns in dataset: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']
Available columns in dataset: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']
Train / Dev / Test sizes (after subsampling): 5000 / 500 / 500
Epoch 1, step 50, t=2, loss=0.7757
Epoch 1, step 100, t=5, loss=2.0129
Epoch 1, step 150, t=2, loss=0.2736
Epoch 1, step 200, t=4, loss=0.6887
Epoch 1, step 250, t=5, loss=1.0209
Epoch 1, step 300, t=1, loss=0.0412
Training finished.
Normalized EM on test set: 0.0000 over 500 exampl